# GigaGraph 3.2B: SOTA-Grade LLM Pre-Training
### Cold Start on FineWeb-Edu (10B Tokens) | Mixed Knowledge & Logic

**Hardware Target:** Kaggle Dual T4 (2x16GB) | Distributed Sharded Execution
**Architecture:** v8.1 (Positional Embeddings, EBA, Layer-Parallel APTP)


In [1]:
# 1. Environment & Auth
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

from dotenv import load_dotenv
from kaggle_secrets import UserSecretsClient
load_dotenv()

try:
    sc = UserSecretsClient()
    hf_token = sc.get_secret("HF_TOKEN")
    wandb_key = sc.get_secret("WANDB_API_KEY")
except:
    hf_token, wandb_key = os.getenv('HF_TOKEN'), os.getenv('WANDB_API_KEY')

! [ -f pyproject.toml ] || uv init --no-workspace
!uv add torch datasets transformers wandb python-dotenv tqdm huggingface_hub

import torch, wandb
from huggingface_hub import login
if hf_token: login(token=hf_token)
if wandb_key: wandb.login(key=wandb_key)


In [2]:
# 2. Reconstruct v8.1 Scaling Modules
from aptp_gnn import GigaGraph_3B
from data_pipeline import GigaDataPipeline
from tqdm import tqdm

In [3]:
# 3. GigaGraph 3.2B Training Loop
VOCAB_SIZE = 128256  # Llama-3
D_MODEL = 3072
DEPTH = 32
BATCH_SIZE = 2
ACCUM_STEPS = 64
SEQ_LEN = 1024
LEARNING_RATE = 1e-4

model = GigaGraph_3B(vocab_size=VOCAB_SIZE, depth=DEPTH, d_model=D_MODEL)
pipeline = GigaDataPipeline()
loader = pipeline.get_dataloader(batch_size=BATCH_SIZE, seq_len=SEQ_LEN)

wandb.init(project="gigagraph-3b-cold-start")

for i, batch in enumerate(tqdm(loader)):
    x = batch.to("cuda:0")
    y = torch.roll(x, -1, dims=1)
    loss = model.train_step(x, y, lr=LEARNING_RATE)
    if i % 10 == 0:
        wandb.log({"loss": loss.item()})
wandb.finish()